In [2]:
import pandas as pd

import os

cwd = os.getcwd()
parent = os.path.dirname(cwd)
grandparent = os.path.dirname(parent)

data_dir =  f"{parent}/data/"

mitochondria_prot = pd.read_csv(data_dir + "subcell_location_Mitochondria.tsv", sep="\t")
cytosol_prot = pd.read_csv(data_dir + "subcell_location_Aggresome_Cytosol_Cytoplasmic.tsv", sep="\t")

In [7]:
#!python -m pip install requests
!python -m pip install Bio

     -------------------------------------- 321.4/321.4 KB 6.6 MB/s eta 0:00:00
     ---------------------------------------- 67.2/67.2 KB 3.6 MB/s eta 0:00:00
     ---------------------------------------- 2.7/2.7 MB 7.3 MB/s eta 0:00:00
     ---------------------------------------- 78.3/78.3 KB 4.3 MB/s eta 0:00:00
     ---------------------------------------- 52.0/52.0 KB 2.6 MB/s eta 0:00:00
     ---------------------------------------- 73.5/73.5 KB ? eta 0:00:00
     -------------------------------------- 123.5/123.5 KB 7.1 MB/s eta 0:00:00
     ---------------------------------------- 78.8/78.8 KB ? eta 0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should consider upgrading via the 'c:\Users\jolle\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [20]:
import requests

url = "https://rest.uniprot.org/uniprotkb/stream?compressed=false&format=fasta&query=%28organism_id%3A9606%29%20AND%20%28reviewed%3Atrue%29"

response = requests.get(url)

with open("human_sprot.fasta", "w") as f:
    f.write(response.text)

print("Downloaded human_sprot.fasta")



Downloaded human_sprot.fasta


In [21]:
# Build a UniProt Dictionary
from Bio import SeqIO

seq_dict = {}

seq_length = 50

for record in SeqIO.parse("human_sprot.fasta", "fasta"):
    accession = record.id.split("|")[1]
    seq_dict[accession] = str(record.seq)[:seq_length]

print(f"Loaded {len(seq_dict)} protein sequences")

Loaded 20431 protein sequences


In [22]:
mitochondria_prot["Uniprot_clean"] = (
    mitochondria_prot["Uniprot"]
    .astype(str)
    .str.split("-")
    .str[0]
)

cytosol_prot["Uniprot_clean"] = (
    cytosol_prot["Uniprot"]
    .astype(str)
    .str.split("-")
    .str[0]
)

mitochondria_prot["AA_sequence_50"] = (
    mitochondria_prot["Uniprot_clean"]
    .map(seq_dict)
)

cytosol_prot["AA_sequence_50"] = (
    cytosol_prot["Uniprot_clean"]
    .map(seq_dict)
)

In [25]:
print(
    mitochondria_prot[
        ["Gene", "Uniprot", "AA_sequence_50"]
    ].head()
)

print(
    f"Mitochondrial matches: "
    f"{mitochondria_prot['AA_sequence_50'].notna().sum()} / {len(mitochondria_prot)}"
)

print(
    f"Cytosolic matches: "
    f"{cytosol_prot['AA_sequence_50'].notna().sum()} / {len(cytosol_prot)}"
)

     Gene Uniprot                                     AA_sequence_50
0  A4GALT  Q9NPC4  MSKPPDLLLRLLRGAPRQRVCTLFIIGFKFTFFVSIMIYWHVVGEP...
1   AARS2  Q5JTZ9  MAASVAAAARRLRRAIRRSPAWRGLSHRPLSSEPPAAKASAVRAAF...
2    AASS  Q9UDR5  MLQVHRTGLGRLGVSLSKGLHHKAVLAVRREDVNAWERRAPLAPKH...
3    AATK  Q6ZMQ8  MSSSFFNPSFAFSSHFDPDGAPLSELSWPSSLAVVAVSFSGLFAVI...
4    ABAT  P80404  MASMLLAQRLACSFQHSYRLLVPGSRHISQAAAKVDVEFDYDGPLM...
Mitochondrial matches: 1112 / 1132
Cytosolic matches: 5241 / 5341


In [11]:
full_seq_dict = {}

for record in SeqIO.parse("human_sprot.fasta", "fasta"):
    accession = record.id.split("|")[1]
    full_seq_dict[accession] = str(record.seq)

mitochondria_prot["AA_sequence"] = mitochondria_prot["Uniprot_clean"].map(full_seq_dict)
mitochondria_prot["AA_sequence_40"] = mitochondria_prot["AA_sequence"].str[:40]

In [31]:
#mitochondria_prot
#cytosol_prot[AA_s]
f = open("../data/signalpeptide.csv", "w")
f.write("sequence, location \n")
for s in mitochondria_prot['AA_sequence_50']:
    if str(s) != "nan":
        f.write(f"{s},1 \n")

for s in cytosol_prot['AA_sequence_50']:
    if str(s) != "nan":
        f.write(f"{s},0 \n")

f.close()